In [1]:
from dataclasses import dataclass, field
from pathlib import Path
import json
import time
from typing import Any, Protocol

import pandas as pd
import yaml
from tqdm import tqdm

from kibad_crawler.config import RAW_DATA_DIR, MAX_NUM_BATCH_PAPERS
from kibad_crawler.papers import PaperSearch, get_papers
from kibad_crawler.semantic_scholar import datasets
from kibad_crawler.semantic_scholar import academic_graph
from kibad_crawler.utils import chunker

2026-09-03 15:22:53.803 | INFO     | kibad_crawler.config:<module>:12 - PROJ_ROOT path is: /Users/darenasc/projects/dfki/kibad-crawler


In [2]:
release_id = datasets.get_latest_release()
print(release_id)

2026-08-25


# Publication Venues

In [3]:
links_publication_venues = datasets.download_links(release_id, "publication-venues")
time.sleep(2)
output_path = datasets.download_dataset_from_semantic_scholar(
    url=links_publication_venues["files"][0],
    output_path=RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "publication_venues",
)
df_venues = datasets.read_publication_venues(file_path=output_path)

2026-09-03 15:22:54.124 | INFO     | kibad_crawler.semantic_scholar.datasets:download_links:89 - links - Downloading links from: https://api.semanticscholar.org/datasets/v1/release/2026-08-25/dataset/publication-venues
2026-09-03 15:23:03.184 | INFO     | kibad_crawler.semantic_scholar.datasets:download_links:89 - links - Downloading links from: https://api.semanticscholar.org/datasets/v1/release/2026-08-25/dataset/publication-venues
2026-09-03 15:23:06.738 | INFO     | kibad_crawler.semantic_scholar.datasets:download_links:92 - links - 1 links for `publication-venues` dataset
2026-09-03 15:23:08.742 | INFO     | kibad_crawler.semantic_scholar.datasets:download_dataset_from_semantic_scholar:163 - /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/publication_venues/20260828_124149_00230_i7c6p_271de25e-5b91-4b3c-9b19-3a425599ffcf.gz already downloaded.
2026-09-03 15:23:09.526 | INFO     | kibad_crawler.semantic_scholar.datasets:read_publication_venu

In [4]:
df_venues.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194588 entries, 0 to 194587
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               194588 non-null  object
 1   alternate_issns  194588 non-null  object
 2   alternate_names  194588 non-null  object
 3   alternate_urls   194588 non-null  object
 4   issn             180872 non-null  object
 5   name             194588 non-null  object
 6   type             105597 non-null  object
 7   url              113315 non-null  object
dtypes: object(8)
memory usage: 11.9+ MB


In [5]:
path_journals = Path("/Users/darenasc/projects/dfki/kibad-crawler/data/raw/journals.csv")


def get_issn_codes(path_journals_csv: Path) -> list:
    """Return a list of ISSN code to filter journals. It reads a CSV with ISSN
    column names and filter it's content.

    Args:
        file_path (Path): CSV file

    Returns:
        list: _description_
    """
    df_journals = pd.read_csv(path_journals_csv)
    columns = [x for x in df_journals.columns if "issn" in str(x).lower()]
    issn = []
    for column in columns:
        issn += [x for x in df_journals[column].unique() if isinstance(x, str)]
    return issn


issn = get_issn_codes(path_journals)
publication_venue_ids = df_venues[df_venues["issn"].isin(issn)].id.unique()
print(issn[:5])
print(len(publication_venue_ids))

['1365-2486', '1932-6203', '0167-8809', '0006-3207', '1365-2664']
95


In [6]:
search_params = PaperSearch(publication_venue_ids=publication_venue_ids)
len(search_params.publication_venue_ids)

95

# Papers

In [7]:
links_papers = datasets.download_links(release_id=release_id, dataset_name="papers")

2026-09-03 15:23:09.725 | INFO     | kibad_crawler.semantic_scholar.datasets:download_links:89 - links - Downloading links from: https://api.semanticscholar.org/datasets/v1/release/2026-08-25/dataset/papers
2026-09-03 15:23:13.652 | INFO     | kibad_crawler.semantic_scholar.datasets:download_links:92 - links - 60 links for `papers` dataset


In [8]:
output_path_papers = RAW_DATA_DIR / "semantic_scholar" / "release" / release_id / "papers"

In [9]:
for url in links_papers["files"][:4]:
    get_papers(url=url, output_path=output_path_papers, search_params=search_params)

2026-09-03 15:23:19.787 | DEBUG    | kibad_crawler.papers:get_papers:62 - Skipping existing /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers/20260828_070435_00070_vxskr_03c47c10-86ac-4560-b2f2-5acbad510b08.gz.csv
2026-09-03 15:23:19.910 | INFO     | kibad_crawler.papers:get_abstracts:105 - 4,882 abstracts to download


2026-09-03 15:24:13.384 | DEBUG    | kibad_crawler.papers:get_papers:62 - Skipping existing /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers/20260828_070435_00070_vxskr_0ad137e4-5397-432c-9b26-5926ecf12afe.gz.csv
2026-09-03 15:24:13.445 | INFO     | kibad_crawler.papers:get_abstracts:105 - 7,341 abstracts to download


2026-09-03 15:26:42.971 | DEBUG    | kibad_crawler.papers:get_papers:62 - Skipping existing /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers/20260828_070435_00070_vxskr_161d9e4b-75e1-4d70-96f5-5d9eaf5b7f4f.gz.csv
2026-09-03 15:26:43.037 | INFO     | kibad_crawler.papers:get_abstracts:105 - 7,450 abstracts to download


2026-09-03 15:28:12.467 | DEBUG    | kibad_crawler.papers:get_papers:62 - Skipping existing /Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers/20260828_070435_00070_vxskr_180c5e76-641b-488c-83b6-ca1dbe75d920.gz.csv
2026-09-03 15:28:12.509 | INFO     | kibad_crawler.papers:get_abstracts:105 - 4,961 abstracts to download


# Abstracts

In [10]:
paper_metadata_files = [x for x in output_path_papers.glob("**/*.csv")]

In [11]:
output_path_papers

PosixPath('/Users/darenasc/projects/dfki/kibad-crawler/data/raw/semantic_scholar/release/2026-08-25/papers')

In [12]:
# paper_files = [x for x in (output_path_papers).glob("*.csv")]
# dfs = [pd.read_csv(f) for f in paper_files]
# df_all_papers = pd.concat(dfs)

In [35]:
def get_abstracts(path: Path):
    df = pd.read_csv(path)
    temp_folder = path.name[:-7]
    paper_ids = []
    for i, r in df.iterrows():
        paper_ids.append(r["url"].split("/")[-1])

    json_path = output_path_papers / temp_folder / "json"
    json_path.mkdir(parents=True, exist_ok=True)

    for i, group in tqdm(
        enumerate(chunker(paper_ids, MAX_NUM_BATCH_PAPERS)),
        total=int(len(paper_ids) / MAX_NUM_BATCH_PAPERS),
        desc="Downloading metadata",
    ):
        response_batch = academic_graph.batch(group)

        json_file = json_path / f"papers-metadata-{i}.json"
        with open(str(json_file), "w", encoding="utf-8") as f:
            json.dump(response_batch, f)

In [36]:
get_abstracts(paper_metadata_files[1])

In [ ]:
df = pd.read_csv(paper_metadata_files[0])
temp_folder = paper_metadata_files[0].name[:-7]
paper_ids = []
for i, r in df.iterrows():
    paper_ids.append(r["url"].split("/")[-1])

In [30]:
len(paper_ids)

7450

In [31]:
json_path = output_path_papers / temp_folder / "json"
json_path.mkdir(parents=True, exist_ok=True)

In [32]:
for i, group in tqdm(
    enumerate(chunker(paper_ids, MAX_NUM_BATCH_PAPERS)),
    total=int(len(paper_ids) / MAX_NUM_BATCH_PAPERS),
    desc="Downloading metadata",
):
    # response_batch = self.download_bulk_paper(group)
    response_batch = academic_graph.batch(group)

    json_file = json_path / f"papers-metadata-{i}.json"
    with open(str(json_file), "w", encoding="utf-8") as f:
        json.dump(response_batch, f)

# Relevance Filter

In [ ]:
file_path = "/Users/darenasc/projects/dfki/kibad-crawler/notebooks/query_strings.yml"
with open(file_path, "r") as file:
    yaml_data = yaml.safe_load(file)
    # words = set()
    # for word_list in yaml_data.get("words", []):
    #     words.update(word_list)

# yaml_data

In [ ]:
yaml_data["Biodiversity"]["words"]

In [ ]:
@dataclass
class Concept(Protocol):
    name: str
    words: list[str]

    def check(self, text: str) -> bool: ...


def get_biodiversity_words() -> list[str]:
    return ["species richness", "species number", "number of species"]


@dataclass
class ConceptBiodiversity:
    name: str = "Biodiversity"
    words: list[str] = field(default_factory=get_biodiversity_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False

    def __call__(self, *args: Any, **kwds: Any) -> Any:
        return self.check(*args)


def get_deutschland_words() -> list[str]:
    return ["German", "central europ", "Lower Saxony"]


@dataclass
class ConceptDeutschland:
    name: str = "Deutschland"
    words: list[str] = field(default_factory=get_deutschland_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False


def get_lebensraum_words() -> list[str]:
    return ["Forest", "wood", "stock of trees"]


@dataclass
class ConceptLebensraum:
    name: str = "Lebensraum"
    words: list[str] = field(default_factory=get_lebensraum_words)

    def check(self, text: str) -> bool:
        if any(word in text for word in self.words):
            return True
        else:
            return False


def get_wald_trends_words():
    return ["trend", "chang", "decreas"]


@dataclass
class ConceptWaldTrends:
    name: str = "WaldTrends"
    words: list = field(default_factory=get_wald_trends_words)

    def check(self, text: str) -> bool:
        concept_biodiversity = ConceptBiodiversity()
        concept_deutschland = ConceptDeutschland()
        concept_lebensraum = ConceptLebensraum()
        check_wald_trend = any(word in text for word in self.words)
        return (
            concept_biodiversity.check(text=text)
            and concept_deutschland.check(text=text)
            and concept_lebensraum.check(text=text)
            and check_wald_trend
        )

In [ ]:
concept_wald_trends = ConceptWaldTrends()
print(concept_wald_trends.check("richness trend"))
print(concept_wald_trends.check("species richness trend"))
print(concept_wald_trends.check("species richness trend German Forest"))

In [ ]:
# query._check("richness")
# concept_biodiversity("richness")
# concept_deutschland("richness")

In [ ]:
# @dataclass
# class Concept(Protocol):
#     name: str
#     words: list[str]

#     def _check(self, text: str) -> bool:
#         if any(word in text for word in self.words):
#             return True
#         else:
#             return False

#     def __call__(self, *args: Any, **kwds: Any) -> Any:
#         return self._check(*args)